# 02 - Data Profiling

## Objective

This notebook evaluates the structure, completeness, uniqueness, and basic quality of the raw flight dataset before exploratory analysis and data cleaning.

The profiling process includes:
- Reviewing dataset dimensions and schema
- Measuring missing values
- Identifying duplicate records
- Reviewing distinct values in key categorical columns
- Summarizing numerical columns
- Detecting basic data-quality issues

#### Load configuration and dataset

In [0]:
# Load the project configuration and raw table

from config import project_config as cfg
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

print("Project configuration loaded successfully.")

df_raw = spark.read.table(cfg.RAW_TABLE)

print(f"Table loaded successfully: {cfg.RAW_TABLE}")

#### Review dataset dimensions

In [0]:
# Calculate the dataset dimensions

row_count = df_raw.count()
column_count = len(df_raw.columns)

if row_count == 0:
    raise RuntimeError(f"Table '{cfg.RAW_TABLE}' is empty.")

print(f"Total records: {row_count:,}")
print(f"Total columns: {column_count}")

#### Review dataset schema

In [0]:
# Review column names and inferred data types

print("Dataset schema:")

df_raw.printSchema()

#### Calculate missing-value statistics

In [0]:
# Calculate null counts for every column

null_counts_row = (
    df_raw
    .select(
        [
            F.sum(
                F.col(column_name).isNull().cast("long")
            ).alias(column_name)
            for column_name in df_raw.columns
        ]
    )
    .first()
)

null_profile_data = [
    (
        field.name,
        field.dataType.simpleString(),
        int(null_counts_row[field.name] or 0),
        float((null_counts_row[field.name] or 0) / row_count * 100)
    )
    for field in df_raw.schema.fields
]

null_profile_df = spark.createDataFrame(
    null_profile_data,
    schema=[
        "column_name",
        "data_type",
        "null_count",
        "null_percentage"
    ]
)

#### Display missing-value profile

In [0]:
# Display columns ordered by missing-value percentage

display(
    null_profile_df
    .withColumn(
        "null_percentage",
        F.round(F.col("null_percentage"), 2)
    )
    .orderBy(
        F.col("null_percentage").desc(),
        F.col("column_name")
    )
)

#### Summarize columns containing missing values

In [0]:
# Summarize the number of columns containing null values

columns_with_nulls = null_profile_df.filter(
    F.col("null_count") > 0
).count()

complete_columns = column_count - columns_with_nulls

print(f"Columns with null values: {columns_with_nulls}")
print(f"Columns without null values: {complete_columns}")

#### Duplicate records 

In [0]:
# Identify exact duplicate records

distinct_row_count = df_raw.distinct().count()
duplicate_row_count = row_count - distinct_row_count

print(f"Distinct records: {distinct_row_count:,}")
print(f"Duplicate records: {duplicate_row_count:,}")
print(
    f"Duplicate percentage: "
    f"{(duplicate_row_count / row_count * 100):.4f}%"
)

#### Numerical summary

In [0]:
# Identify numerical columns

numeric_columns = [
    field.name
    for field in df_raw.schema.fields
    if isinstance(field.dataType, NumericType)
]

print(f"Numerical columns found: {len(numeric_columns)}")

display(
    df_raw
    .select(numeric_columns)
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    )
)

#### Key categorical columns

In [0]:
# Review distinct values in important categorical columns

available_categorical_columns = [
    column_name
    for column_name in cfg.KEY_CATEGORICAL_COLUMNS
    if column_name in df_raw.columns
]

for column_name in available_categorical_columns:
    distinct_count = (
        df_raw
        .select(column_name)
        .distinct()
        .count()
    )

    print(f"{column_name}: {distinct_count:,} distinct values")

#### Key categorical distributions

In [0]:
# Display the most frequent values in key categorical columns

for column_name in available_categorical_columns:
    print(f"Distribution for {column_name}:")

    display(
        df_raw
        .groupBy(column_name)
        .count()
        .orderBy(F.col("count").desc())
        .limit(20)
    )

#### Basic quality checks

In [0]:
# Evaluate basic data-quality conditions

quality_checks = []

if "YEAR" in df_raw.columns:
    quality_checks.append(
        (
            "Invalid YEAR values",
            df_raw.filter(
                F.col("YEAR").isNull()
                | (F.col("YEAR") < 2000)
            ).count()
        )
    )

if "MONTH" in df_raw.columns:
    quality_checks.append(
        (
            "Invalid MONTH values",
            df_raw.filter(
                F.col("MONTH").isNull()
                | ~F.col("MONTH").between(1, 12)
            ).count()
        )
    )

if "DAY_OF_MONTH" in df_raw.columns:
    quality_checks.append(
        (
            "Invalid DAY_OF_MONTH values",
            df_raw.filter(
                F.col("DAY_OF_MONTH").isNull()
                | ~F.col("DAY_OF_MONTH").between(1, 31)
            ).count()
        )
    )

if "CANCELLED" in df_raw.columns:
    quality_checks.append(
        (
            "Invalid CANCELLED values",
            df_raw.filter(
                F.col("CANCELLED").isNull()
                | ~F.col("CANCELLED").isin(0, 1)
            ).count()
        )
    )

if "DIVERTED" in df_raw.columns:
    quality_checks.append(
        (
            "Invalid DIVERTED values",
            df_raw.filter(
                F.col("DIVERTED").isNull()
                | ~F.col("DIVERTED").isin(0, 1)
            ).count()
        )
    )

quality_check_df = spark.createDataFrame(
    quality_checks,
    ["quality_check", "invalid_record_count"]
)

display(quality_check_df)

#### Preview sample records

In [0]:
# Preview sample records from the raw dataset

display(df_raw.limit(10))

#### Complete profiling

In [0]:
print("Data profiling completed successfully.")